### 붓꽃(Iris) 데이터 셋을 이용한 클러스터 평가

In [ ]:
# 필요한 라이브러리를 불러온다.
from sklearn.preprocessing import scale
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans

# 실루엣 분석 metric 값을 구하기 위한 함수를 불러온다.
from sklearn.metrics import silhouette_samples, silhouette_score

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 주피터 노트북에서 그래프를 바로 출력하기 위한 설정이다.
%matplotlib inline

# 붓꽃 데이터를 불러온다.
iris = load_iris()

# feature 이름을 지정한다.
feature_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

# 붓꽃 데이터를 DataFrame으로 변환한다.
irisDF = pd.DataFrame(data=iris.data, columns=feature_names)

# KMeans 객체를 생성하고 군집 개수를 3개로 하여 군집화를 수행한다.
kmeans = KMeans(n_clusters=3, init='k-means++', max_iter=300, random_state=0).fit(irisDF)

# 군집화 결과 레이블을 cluster 컬럼으로 추가한다.
irisDF['cluster'] = kmeans.labels_

# 붓꽃 데이터의 각 개별 데이터에 대한 실루엣 계수를 계산한다.
score_samples = silhouette_samples(iris.data, irisDF['cluster'])

# 반환된 실루엣 계수 배열의 shape를 출력한다.
print('silhouette_samples( ) return 값의 shape', score_samples.shape)

# 계산된 실루엣 계수를 silhouette_coeff 컬럼으로 추가한다.
irisDF['silhouette_coeff'] = score_samples

# 전체 데이터의 평균 실루엣 계수를 계산한다.
average_score = silhouette_score(iris.data, irisDF['cluster'])

# 평균 실루엣 계수를 출력한다.
print('붓꽃 데이터셋 Silhouette Analysis Score:{0:.3f}'.format(average_score))

# 상위 3개 데이터를 확인한다.
irisDF.head(3)

In [ ]:
# 각 클러스터별 평균 실루엣 계수를 계산하여 확인한다.
irisDF.groupby('cluster')['silhouette_coeff'].mean()

### 클러스터별 평균 실루엣 계수의 시각화를 통한 클러스터 개수 최적화 방법

In [ ]:
# 여러 개의 클러스터 개수를 리스트로 입력받아
# 각각의 실루엣 계수를 면적 그래프로 시각화하는 함수를 정의한다.
def visualize_silhouette(cluster_lists, X_features):

    # 함수 내부에서 필요한 라이브러리를 불러온다.
    from sklearn.datasets import make_blobs
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_samples, silhouette_score

    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import math

    # 입력값으로 클러스터 개수들을 리스트로 받아
    # 각 개수별로 클러스터링을 수행하고 실루엣 계수를 계산한다.
    n_cols = len(cluster_lists)

    # 리스트에 있는 클러스터 개수만큼 subplot을 생성한다.
    fig, axs = plt.subplots(figsize=(4 * n_cols, 4), nrows=1, ncols=n_cols)

    # 각 클러스터 개수에 대해 반복하면서 실루엣 계수를 시각화한다.
    for ind, n_cluster in enumerate(cluster_lists):

        # KMeans 군집화를 수행하고 군집 레이블을 예측한다.
        clusterer = KMeans(n_clusters=n_cluster, max_iter=500, random_state=0)
        cluster_labels = clusterer.fit_predict(X_features)

        # 평균 실루엣 계수와 각 데이터의 실루엣 계수를 계산한다.
        sil_avg = silhouette_score(X_features, cluster_labels)
        sil_values = silhouette_samples(X_features, cluster_labels)

        # y축 시작 위치를 설정한다.
        y_lower = 10

        # 그래프 제목과 축 정보를 설정한다.
        axs[ind].set_title('Number of Cluster : ' + str(n_cluster) + '\n'
                           'Silhouette Score :' + str(round(sil_avg, 3)))
        axs[ind].set_xlabel("The silhouette coefficient values")
        axs[ind].set_ylabel("Cluster label")
        axs[ind].set_xlim([-0.1, 1])
        axs[ind].set_ylim([0, len(X_features) + (n_cluster + 1) * 10])
        axs[ind].set_yticks([])  # y축 눈금 제거
        axs[ind].set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1])

        # 클러스터 개수별로 fill_betweenx() 형태의 막대 그래프를 그린다.
        for i in range(n_cluster):
            ith_cluster_sil_values = sil_values[cluster_labels == i]
            ith_cluster_sil_values.sort()

            size_cluster_i = ith_cluster_sil_values.shape[0]
            y_upper = y_lower + size_cluster_i

            color = cm.nipy_spectral(float(i) / n_cluster)
            axs[ind].fill_betweenx(
                np.arange(y_lower, y_upper),
                0,
                ith_cluster_sil_values,
                facecolor=color,
                edgecolor=color,
                alpha=0.7
            )

            # 각 클러스터 번호를 그래프 왼쪽에 표시한다.
            axs[ind].text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

            # 다음 클러스터를 위해 y축 시작 위치를 조정한다.
            y_lower = y_upper + 10

        # 평균 실루엣 계수 위치를 빨간 점선으로 표시한다.
        axs[ind].axvline(x=sil_avg, color="red", linestyle="--")

In [ ]:
# make_blobs를 이용하여
# 4개의 클러스터 중심을 갖는 500개의 2차원 데이터를 생성한다.
from sklearn.datasets import make_blobs

X, y = make_blobs(
    n_samples=500,
    n_features=2,
    centers=4,
    cluster_std=1,
    center_box=(-10.0, 10.0),
    shuffle=True,
    random_state=1
)

# 클러스터 개수를 2, 3, 4, 5로 바꾸어가며
# 클러스터별 실루엣 계수 평균값을 시각화한다.
visualize_silhouette([2, 3, 4, 5], X)

In [ ]:
# 붓꽃 데이터를 다시 불러온다.
from sklearn.datasets import load_iris

iris = load_iris()

# 붓꽃 데이터에 대해 클러스터 개수를 2, 3, 4, 5로 바꾸어가며
# 실루엣 계수를 시각화한다.
visualize_silhouette([2, 3, 4, 5], iris.data)